In [ ]:
from collections import OrderedDict
import pandas as pd

In [ ]:


# ============================================================
# Raw layer-wise pruning stats from your text
# ============================================================

DATA = OrderedDict({
    "20%": OrderedDict({
        "features.0":   {"zeros": 7,        "total": 1728},
        "features.2":   {"zeros": 1081,     "total": 36864},
        "features.5":   {"zeros": 2228,     "total": 73728},
        "features.7":   {"zeros": 5138,     "total": 147456},
        "features.10":  {"zeros": 11816,    "total": 294912},
        "features.12":  {"zeros": 29149,    "total": 589824},
        "features.14":  {"zeros": 28450,    "total": 589824},
        "features.17":  {"zeros": 67539,    "total": 1179648},
        "features.19":  {"zeros": 169014,   "total": 2359296},
        "features.21":  {"zeros": 168863,   "total": 2359296},
        "features.24":  {"zeros": 158506,   "total": 2359296},
        "features.26":  {"zeros": 157673,   "total": 2359296},
        "features.28":  {"zeros": 177098,   "total": 2359296},
        "classifier.0": {"zeros": 23945866, "total": 102760448},
        "classifier.3": {"zeros": 1932135,  "total": 16777216},
        "classifier.6": {"zeros": 3255,     "total": 40960},
    }),
    "40%": OrderedDict({
        "features.0":   {"zeros": 21,       "total": 1728},
        "features.2":   {"zeros": 2268,     "total": 36864},
        "features.5":   {"zeros": 4640,     "total": 73728},
        "features.7":   {"zeros": 10673,    "total": 147456},
        "features.10":  {"zeros": 25239,    "total": 294912},
        "features.12":  {"zeros": 60999,    "total": 589824},
        "features.14":  {"zeros": 59848,    "total": 589824},
        "features.17":  {"zeros": 142060,   "total": 1179648},
        "features.19":  {"zeros": 354532,   "total": 2359296},
        "features.21":  {"zeros": 354568,   "total": 2359296},
        "features.24":  {"zeros": 332611,   "total": 2359296},
        "features.26":  {"zeros": 330446,   "total": 2359296},
        "features.28":  {"zeros": 370371,   "total": 2359296},
        "classifier.0": {"zeros": 47667071, "total": 102760448},
        "classifier.3": {"zeros": 3993353,  "total": 16777216},
        "classifier.6": {"zeros": 6935,     "total": 40960},
    }),
    "60%": OrderedDict({
        "features.0":   {"zeros": 38,       "total": 1728},
        "features.2":   {"zeros": 3682,     "total": 36864},
        "features.5":   {"zeros": 7663,     "total": 73728},
        "features.7":   {"zeros": 17852,    "total": 147456},
        "features.10":  {"zeros": 41928,    "total": 294912},
        "features.12":  {"zeros": 101090,   "total": 589824},
        "features.14":  {"zeros": 99250,    "total": 589824},
        "features.17":  {"zeros": 234522,   "total": 1179648},
        "features.19":  {"zeros": 584120,   "total": 2359296},
        "features.21":  {"zeros": 583638,   "total": 2359296},
        "features.24":  {"zeros": 547764,   "total": 2359296},
        "features.26":  {"zeros": 545263,   "total": 2359296},
        "features.28":  {"zeros": 607530,   "total": 2359296},
        "classifier.0": {"zeros": 70773910, "total": 102760448},
        "classifier.3": {"zeros": 6413753,  "total": 16777216},
        "classifier.6": {"zeros": 11450,    "total": 40960},
    }),
    "80%": OrderedDict({
        "features.0":   {"zeros": 68,       "total": 1728},
        "features.2":   {"zeros": 6043,     "total": 36864},
        "features.5":   {"zeros": 12604,    "total": 73728},
        "features.7":   {"zeros": 29311,    "total": 147456},
        "features.10":  {"zeros": 68456,    "total": 294912},
        "features.12":  {"zeros": 165538,   "total": 589824},
        "features.14":  {"zeros": 162502,   "total": 589824},
        "features.17":  {"zeros": 381736,   "total": 1179648},
        "features.19":  {"zeros": 942114,   "total": 2359296},
        "features.21":  {"zeros": 943486,   "total": 2359296},
        "features.24":  {"zeros": 885674,   "total": 2359296},
        "features.26":  {"zeros": 882653,   "total": 2359296},
        "features.28":  {"zeros": 970959,   "total": 2359296},
        "classifier.0": {"zeros": 92092848, "total": 102760448},
        "classifier.3": {"zeros": 9868991,  "total": 16777216},
        "classifier.6": {"zeros": 18287,    "total": 40960},
    }),
})

# ============================================================
# Region definitions
# ============================================================

REGIONS = OrderedDict({
    "Early conv": ["features.0", "features.2", "features.5", "features.7"],
    "Mid conv":   ["features.10", "features.12", "features.14"],
    "Late conv":  ["features.17", "features.19", "features.21", "features.24", "features.26", "features.28"],
    "Classifier": ["classifier.0", "classifier.3", "classifier.6"],
})

REGIONS_2WAY = OrderedDict({
    "All features": [k for k in DATA["20%"].keys() if k.startswith("features.")],
    "Classifier":   [k for k in DATA["20%"].keys() if k.startswith("classifier.")],
})


# ============================================================
# Helpers
# ============================================================

def layer_sparsity(layer_stats):
    return layer_stats["zeros"] / layer_stats["total"]

def weighted_region_sparsity(pruning_stats, layers):
    zeros = sum(pruning_stats[layer]["zeros"] for layer in layers)
    total = sum(pruning_stats[layer]["total"] for layer in layers)
    return zeros / total

def unweighted_region_sparsity(pruning_stats, layers):
    vals = [layer_sparsity(pruning_stats[layer]) for layer in layers]
    return sum(vals) / len(vals)

def model_total_params(pruning_stats):
    return sum(v["total"] for v in pruning_stats.values())

def model_total_zeros(pruning_stats):
    return sum(v["zeros"] for v in pruning_stats.values())

def region_param_share(pruning_stats, layers):
    total_region = sum(pruning_stats[layer]["total"] for layer in layers)
    total_model = model_total_params(pruning_stats)
    return total_region / total_model

def pruning_concentration(pruning_stats, layers):
    zeros_region = sum(pruning_stats[layer]["zeros"] for layer in layers)
    zeros_model = model_total_zeros(pruning_stats)
    return zeros_region / zeros_model

def as_percent(x, digits=2):
    return f"{100*x:.{digits}f}%"

def make_table_from_region_metric(data, regions, metric_fn, digits=2):
    rows = []
    for region_name, layers in regions.items():
        row = {"Region": region_name}
        for sparsity_name, pruning_stats in data.items():
            row[sparsity_name] = as_percent(metric_fn(pruning_stats, layers), digits)
        rows.append(row)
    return pd.DataFrame(rows)

def make_parameter_share_table(reference_stats, regions, digits=2):
    rows = []
    for region_name, layers in regions.items():
        rows.append({
            "Region": region_name,
            "Parameter Share of Total Model": as_percent(
                region_param_share(reference_stats, layers), digits
            )
        })
    return pd.DataFrame(rows)

def print_tsv(df, title):
    print(f"\n{title}")
    print(df.to_csv(sep="\t", index=False))


# ============================================================
# Generate the 5 requested tables
# ============================================================

# 1) Weighted pruning by model region
weighted_region_df = make_table_from_region_metric(
    DATA, REGIONS, weighted_region_sparsity, digits=2
)

# 2) Unweighted average layer sparsity by region
unweighted_region_df = make_table_from_region_metric(
    DATA, REGIONS, unweighted_region_sparsity, digits=2
)

# 3) Parameter share of each region
# totals are identical across pruning levels, so use any one reference block
reference_stats = next(iter(DATA.values()))
parameter_share_df = make_parameter_share_table(reference_stats, REGIONS, digits=2)

# 4) Conv vs classifier only
conv_vs_classifier_df = make_table_from_region_metric(
    DATA, REGIONS_2WAY, weighted_region_sparsity, digits=2
)

# 5) Pruning concentration table
pruning_concentration_df = make_table_from_region_metric(
    DATA, REGIONS, pruning_concentration, digits=2
)

# ============================================================
# Driver code: display / export
# ============================================================

if __name__ == "__main__":
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    print_tsv(weighted_region_df, "Weighted pruning by model region")
    print_tsv(unweighted_region_df, "Unweighted average layer sparsity by region")
    print_tsv(parameter_share_df, "Parameter share of each region")
    print_tsv(conv_vs_classifier_df, "Conv vs classifier only")
    print_tsv(pruning_concentration_df, "Pruning concentration")

    # Optional: save to CSV files
    weighted_region_df.to_csv("weighted_pruning_by_region.csv", index=False)
    unweighted_region_df.to_csv("unweighted_avg_layer_sparsity_by_region.csv", index=False)
    parameter_share_df.to_csv("parameter_share_by_region.csv", index=False)
    conv_vs_classifier_df.to_csv("conv_vs_classifier.csv", index=False)
    pruning_concentration_df.to_csv("pruning_concentration.csv", index=False)